# 01 -- Data Acquisition (GDC API + TCGAbiolinks)

This notebook demonstrates how the raw RNA-seq gene expression and clinical
data for this study are obtained from the **NCI Genomic Data Commons (GDC)**.

Two acquisition routes are implemented and both are exercised below:

1. **`src/download.py`** -- a pure-Python client against the public
   [GDC REST API](https://docs.gdc.cancer.gov/API/Users_Guide/Getting_Started/).
   This is the primary, dependency-light route and is demonstrated live below
   against real TCGA-BRCA data.
2. **`scripts/download_tcgabiolinks.R`** -- an equivalent R script using
   [`TCGAbiolinks`](https://bioconductor.org/packages/release/bioc/html/TCGAbiolinks.html),
   the de-facto standard tool in the TCGA/GDC research community, generated by
   `src.download.write_tcgabiolinks_script()`. Prefer this route for a full,
   production-scale pull of an entire project.

**Why a small subset, and why a synthetic cohort powers the rest of the study.**
A full TCGA-BRCA RNA-seq pull is ~1,100 cases and several hundred megabytes --
a legitimate real workload, but impractical to run unattended for a portfolio
demo. We query and download a small **real** subset live (below) to prove the
client works end-to-end against the live GDC API, and then use a
biologically-motivated **synthetic cohort** (`src/simulate.py`) with a known
PAM50-like ground truth for the full, reproducible analysis in notebooks
02-08. This lets us honestly check *whether the unsupervised pipeline
recovers signal that is known, by construction, to be there* -- the same
logic used to benchmark clustering methods against PAM50 calls in the
literature.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / ".." / "src").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.config import load_config
from src.logging_config import setup_logging

setup_logging()
cfg = load_config(REPO_ROOT / "config.yaml")
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results"
REPORTS = REPO_ROOT / "reports"
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
print(f"Repo root: {REPO_ROOT}")
print(f"Cancer type: {cfg['project']['cancer_type']}")


Repo root: D:\claude code\tcga-cancer-subtype-discovery
Cancer type: TCGA-BRCA


## 1.1 Query the GDC `files` endpoint for real TCGA-BRCA RNA-seq files

This is a live network call to `https://api.gdc.cancer.gov`.

In [2]:
from src.config import override
from src.download import query_rna_seq_files

demo_cfg = override(cfg, **{"gdc.max_cases": 8})
files_df = query_rna_seq_files(demo_cfg)
files_df


2026-07-28 15:04:34 | INFO     | src.download | Querying GDC files endpoint for TCGA-BRCA RNA-seq gene expression files...


2026-07-28 15:04:35 | INFO     | src.download | GDC returned 8 candidate files.


2026-07-28 15:04:35 | INFO     | src.download | Selected 8 unique cases after de-duplication / max_cases cap.


,file_id,file_name,case_id,case_submitter_id,sample_type,sample_submitter_id
0,744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...,878f975b-94fd-4d69-b7e7-1ed3ac2ee438,TCGA-BH-A18H,Primary Tumor,TCGA-BH-A18H-01A
1,4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...,e4fc0909-f284-4471-866d-d8967b6adcbc,TCGA-E2-A14P,Primary Tumor,TCGA-E2-A14P-01A
2,1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...,e5aae05a-478e-4a55-a27c-12b2b4be302a,TCGA-AN-A04A,Primary Tumor,TCGA-AN-A04A-01A
3,2d5b0962-b5c4-4482-9f28-47e4dcdb6df6,602e3d24-a45d-4fe4-9b70-a679d93f470d.rna_seq.a...,16fc3677-0393-4ed1-ad3f-c8355f056369,TCGA-5L-AAT1,Primary Tumor,TCGA-5L-AAT1-01A
4,c2515ef2-5cfc-4cc2-9541-ae072616d361,f8ff90c6-3554-4e19-ae56-18f30f51f9b5.rna_seq.a...,d0b78f3f-a198-437a-ab8c-204345d3b75d,TCGA-AQ-A0Y5,Primary Tumor,TCGA-AQ-A0Y5-01A
5,61647238-a8c4-4156-af15-c32947983cfd,4c6cb929-293d-452e-b7f8-d535b48cf073.rna_seq.a...,566792ae-f853-4a47-856d-f02cdcfcb18a,TCGA-A8-A07I,Primary Tumor,TCGA-A8-A07I-01A
6,488399aa-ef3c-42d7-817f-9db11b9902dd,a8e0466b-d106-44b7-b846-c4fb666715f5.rna_seq.a...,b7a4df9a-4e25-4550-9c10-338ca1921c2f,TCGA-E9-A1R4,Primary Tumor,TCGA-E9-A1R4-01A
7,3d0b3b80-92ae-44dc-89fc-2ead925b81a2,eb44c266-8b8f-4012-ad67-f4ac40698391.rna_seq.a...,b9c6c069-2a2b-4cc7-a398-dc45c132d979,TCGA-AC-A8OP,Primary Tumor,TCGA-AC-A8OP-01A


## 1.2 Download one real gene-expression quantification file

GDC STAR-Counts files are gzip TSVs with gene-level unstranded/TPM/FPKM columns.

In [3]:
from src.download import download_expression_file

sample_file_id = files_df.iloc[0]["file_id"]
gene_df = download_expression_file(cfg["gdc"]["base_url"], sample_file_id)
print(gene_df.shape)
gene_df.head(10)


(19962, 8)


,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
gene_id,,,,,,,,
ENSG00000000003.15,TSPAN6,protein_coding,426,213,213,6.7894,2.4124,2.4300
ENSG00000000005.6,TNMD,protein_coding,3,2,1,0.1469,0.0522,0.0526
ENSG00000000419.13,DPM1,protein_coding,1439,722,717,86.1885,30.6240,30.8480
ENSG00000000457.14,SCYL3,protein_coding,1049,861,830,11.0178,3.9148,3.9434
ENSG00000000460.17,C1orf112,protein_coding,326,511,506,3.9477,1.4027,1.4129
ENSG00000000938.13,FGR,protein_coding,1015,526,489,21.6964,7.7090,7.7654
ENSG00000000971.16,CFH,protein_coding,2292,1162,1130,20.7638,7.3777,7.4317
ENSG00000001036.14,FUCA2,protein_coding,1452,1066,1095,37.1968,13.2165,13.3132
ENSG00000001084.13,GCLC,protein_coding,1506,887,890,12.6332,4.4888,4.5216


## 1.3 Query matched clinical metadata

Age, sex, tumor stage, vital status, and follow-up time -- needed for the clinical and survival analyses in notebook 07.

In [4]:
from src.download import query_clinical_metadata

clinical_demo = query_clinical_metadata(files_df["case_submitter_id"].tolist(), cfg)
clinical_demo


2026-07-28 15:04:42 | INFO     | src.download | Querying GDC cases endpoint for clinical metadata on 8 cases...


,case_submitter_id,gender,age_at_index,vital_status,ajcc_pathologic_stage,tumor_grade,primary_diagnosis,days_to_death,days_to_last_follow_up,days_to_event,event_observed
0,TCGA-BH-A18H,female,63,Alive,Stage IA,None,"Infiltrating duct carcinoma, NOS",NaN,652.0,652.0,0
1,TCGA-E2-A14P,female,79,Alive,Stage IIIC,None,"Infiltrating duct carcinoma, NOS",NaN,1246.0,1246.0,0
2,TCGA-AN-A04A,female,36,Alive,Stage IIIA,None,"Infiltrating duct carcinoma, NOS",NaN,90.0,90.0,0
3,TCGA-AQ-A0Y5,female,70,Dead,None,None,"Myelodysplastic syndrome, NOS",172.0,NaN,172.0,1
4,TCGA-5L-AAT1,female,63,Alive,Stage IV,None,"Lobular carcinoma, NOS",NaN,1471.0,1471.0,0
5,TCGA-A8-A07I,female,69,Alive,Stage IIIA,None,"Infiltrating duct carcinoma, NOS",NaN,426.0,426.0,0
6,TCGA-E9-A1R4,female,66,Alive,Stage IA,None,"Infiltrating duct carcinoma, NOS",NaN,186.0,186.0,0
7,TCGA-AC-A8OP,female,72,Alive,Stage IA,None,"Infiltrating duct carcinoma, NOS",NaN,614.0,614.0,0


## 1.4 Full-scale download (not run here)

Running `run_download(cfg)` with `gdc.max_cases: null` in `config.yaml` (or
the CLI: `python -m src.cli download`) will pull every matching TCGA-BRCA
case and assemble a full genes x samples expression matrix plus clinical
table under `data/raw/` and `data/metadata/`. We do not execute the full
pull in this notebook to keep it fast and network-light; see the module
docstring in `src/download.py` for the details.

In [5]:
from src.download import write_tcgabiolinks_script

r_script_path = write_tcgabiolinks_script()
print(f"TCGAbiolinks companion script available at: {r_script_path.relative_to(REPO_ROOT)}")


TCGAbiolinks companion script available at: scripts\download_tcgabiolinks.R


## 1.5 Generate the synthetic demo cohort used for the rest of this study

In [6]:
from src.simulate import run_simulation

expr_path, clinical_path = run_simulation(cfg)
print("Expression matrix:", expr_path)
print("Clinical metadata:", clinical_path)


2026-07-28 15:04:43 | INFO     | src.simulate | Simulated cohort: 500 samples x 4000 genes | subtypes: {'Luminal A': 204, 'Basal-like': 105, 'HER2-enriched': 103, 'Luminal B': 88}


2026-07-28 15:04:45 | INFO     | src.simulate | Wrote simulated expression matrix: D:\claude code\tcga-cancer-subtype-discovery\data\raw\TCGA-BRCA_simulated_expression_matrix.tsv


2026-07-28 15:04:45 | INFO     | src.simulate | Wrote simulated clinical metadata: D:\claude code\tcga-cancer-subtype-discovery\data\metadata\TCGA-BRCA_simulated_clinical.csv


Expression matrix: D:\claude code\tcga-cancer-subtype-discovery\data\raw\TCGA-BRCA_simulated_expression_matrix.tsv
Clinical metadata: D:\claude code\tcga-cancer-subtype-discovery\data\metadata\TCGA-BRCA_simulated_clinical.csv


In [7]:
expr_preview = pd.read_csv(expr_path, sep="\t", index_col=0, nrows=5)
clinical_preview = pd.read_csv(clinical_path, index_col=0)
display(expr_preview.iloc[:, :6])
display(clinical_preview.head())
print(f"\nCohort: {clinical_preview.shape[0]} samples")
print(clinical_preview["true_subtype"].value_counts())


,SIM-BRCA-0000,SIM-BRCA-0001,SIM-BRCA-0002,SIM-BRCA-0003,SIM-BRCA-0004,SIM-BRCA-0005
gene_name,,,,,,
AURKA,199.914828,113.557911,137.781336,28.291655,400.163781,15.973833
BCL2,99.776499,45.829468,18.751182,228.735359,107.031699,161.021763
BIRC5,648.131603,212.583484,314.762634,149.543504,1221.058395,241.184739
CCNB1,376.703168,134.109646,725.097799,42.467035,684.655282,43.463694
CCNE1,76.616978,71.677968,29.942714,49.274188,48.435894,30.106291


,true_subtype,sequencing_plate,age_at_index,gender,ajcc_pathologic_stage,tumor_grade,vital_status,days_to_event,event_observed
case_submitter_id,,,,,,,,,
SIM-BRCA-0000,Luminal B,Plate_C,57.1,female,Stage IIB,G1,Dead,770.0,1
SIM-BRCA-0001,HER2-enriched,Plate_A,65.7,male,Stage IA,G1,Dead,1209.0,1
SIM-BRCA-0002,Basal-like,Plate_B,67.2,female,Stage IIB,G3,Dead,24.0,1
SIM-BRCA-0003,Luminal A,Plate_C,55.5,female,Stage IIA,G1,Alive,2268.0,0
SIM-BRCA-0004,Luminal B,Plate_A,53.4,female,Stage IIIA,G2,Alive,3308.0,0



Cohort: 500 samples
true_subtype
Luminal A        204
Basal-like       105
HER2-enriched    103
Luminal B         88
Name: count, dtype: int64


## Summary

* The GDC API client in `src/download.py` is real and functional -- verified
  live above against `TCGA-BRCA` RNA-seq (STAR-Counts) files and matched
  clinical records.
* A companion R/TCGAbiolinks script is provided for full-scale, production
  pulls (`scripts/download_tcgabiolinks.R`).
* The synthetic cohort generated here (500 samples, 4,000 genes, 4 PAM50-like
  subtypes, correlated clinical/survival data) powers the reproducible
  analysis in the remaining notebooks. See `src/simulate.py` for the full
  biological rationale, including which real HGNC marker genes drive each
  subtype and how survival/stage/grade are coupled to subtype identity.
